#Load Model

In [2]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMBED_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [1]:
import os
import torch
import json
import torch.nn as nn    

In [3]:
save_dir = "models"

In [4]:
_ = torch.manual_seed(123)

In [6]:
file_name = "vocab.json"
with open(os.path.join(save_dir, file_name), "r") as f:
    vocab = json.load(f)

In [7]:
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: He
1: She
2: and
3: art
4: books
5: builds
6: caves.
7: climbs
8: collaborates
9: complex
10: composes
11: creative
12: curates
13: daily.
14: designs
15: digital
16: documents
17: every
18: everyday
19: exhibitions
20: experiments
21: explores
22: fairs.
23: filmmakers.
24: for
25: friends.
26: harmonies
27: her
28: in
29: jewelry
30: local
31: maps
32: marathons.
33: models.
34: mountains
35: music
36: music.
37: navigation
38: nearby
39: newspaper
40: novel
41: novels.
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects.
48: puzzles.
49: reads
50: regularly.
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars.
58: studies
59: teaches
60: the
61: trains
62: trips.
63: tunes
64: using
65: weekend.
66: wildlife
67: with
68: wooden
69: writes


In [8]:
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for idx, word in enumerate(vocab)}

In [9]:
def text_to_token_ids(text, word_to_idx):
    return [word_to_idx[word] for word in text.split()]

def token_ids_to_text(token_ids, idx_to_word):
    return " ".join([idx_to_word[idx] for idx in token_ids])

In [10]:
class Attention(nn.Module):
    def __init__(self, d_in,d_out, context_length):
        super().__init__()

        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length),diagonal=1))
    
    def forward(self, x,return_weights=False):
        _, num_tokens, _ = x.shape
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)

        attn_scores = query @ key.transpose(1, 2)

        attn_weights = torch.softmax(attn_scores /(self.d_out ** 0.5), dim=-1)

        context_vec = attn_weights @ value

        if return_weights:
            return context_vec, attn_weights
        
    
        return context_vec

In [28]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.att = Attention(
            d_in= EMBED_DIM, 
            d_out=EMBED_DIM, 
            context_length=CONTEXT_LEN)
    
    def forward(self, x):
       shortcut = x
       x = self.att(x)
       x = x + shortcut
       return x

In [29]:
class GPTModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMBED_DIM)  # Add positional embedding
        self.trf_block1 = TransformerBlock()
        self.trf_block2 = TransformerBlock()
        self.trf_block3 = TransformerBlock()
        self.trf_block4 = TransformerBlock()
        self.trf_block5 = TransformerBlock()
        self.trf_block6 = TransformerBlock()

        self.output_layer = nn.Linear(EMBED_DIM, VOCAB_SIZE, bias=False)
    
    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # Ensure device compatibility

        x = tok_embeds + pos_embeds
        x = self.trf_block1(x)
        x = self.trf_block2(x)
        x = self.trf_block3(x)
        x = self.trf_block4(x)
        x = self.trf_block5(x)  
        x = self.trf_block6(x) 
       
        logits = self.output_layer(x)
        return logits

In [30]:
model = GPTModel()

In [35]:
file_name = "parameters.bin"
#model.load_state_dict(torch.load(os.path.join(save_dir, file_name)))

state_dict = torch.load(os.path.join(save_dir, file_name))

# Rename keys in the state dictionary
new_state_dict = {}
for key in state_dict:
    new_key = key.replace("trm_block", "trf_block")  # Replace 'trm_block' with 'trf_block'
    new_state_dict[new_key] = state_dict[key]

# Load the updated state dictionary into the model
model.load_state_dict(new_state_dict)

<All keys matched successfully>

In [37]:
_ = model.eval()


In [38]:
def generate(start_text):
    token_ids = text_to_token_ids(start_text, word_to_idx)
    num_new_tokens = CONTEXT_LEN - len(token_ids)
    idx = torch.tensor(token_ids).unsqueeze(0)  # Shape: (1, seq_len)

    for _ in range(num_new_tokens):
        idx_cond = idx[:, -CONTEXT_LEN:]  # Get the last CONTEXT_LEN tokens
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    idx = idx.view(-1).tolist()
    text = token_ids_to_text(idx, idx_to_word)
    return text

In [40]:
text = generate("He reads")
print("Generated text: ",text)

Generated text:  He reads newspaper navigation climbs local


In [42]:
text = generate("She composes")
print("Generated text:",text)

Generated text: She composes songs and curates with
